# 05 - Évaluation et comparaison

Ce notebook compare les performances du modèle baseline sans contexte et du modèle contextuel sur MovieLens 100K.

L'objectif est de mesurer l'apport des features temporo-sessionnelles via des métriques de ranking (Hit Rate@10, NDCG@10, MRR).


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

sys.path.append(str((Path('..') / 'src').resolve()))
from data_utils import load_movielens_100k, encode_ids
from context_features import extract_all_context_features
from models import NCF, NCFContext
from metrics import hit_rate, ndcg, mrr

print('Python:', sys.version.split()[0])
print('torch :', torch.__version__)


Python: 3.12.13
torch : 2.12.0+cu130


## Chargement des modèles sauvegardés

Nous cherchons les poids des modèles baseline et contextuel dans `results/models`.


In [3]:
NOTEBOOK_ROOT = Path('.')
RESULTS_DIR = (NOTEBOOK_ROOT / '..' / 'results' / 'models').resolve()
BASELINE_PATH = RESULTS_DIR / 'baseline_ncf_ml100k.pt'
CONTEXT_PATH = RESULTS_DIR / 'context_ncf_ml100k.pt'

print('BASELINE_PATH =', BASELINE_PATH)
print('CONTEXT_PATH =', CONTEXT_PATH)

baseline_exists = BASELINE_PATH.exists()
context_exists = CONTEXT_PATH.exists()
print('Baseline exists:', baseline_exists)
print('Context exists:', context_exists)


BASELINE_PATH = /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/baseline_ncf_ml100k.pt
CONTEXT_PATH = /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/context_ncf_ml100k.pt
Baseline exists: True
Context exists: True


## Préparation du test set

Nous chargeons MovieLens 100K et appliquons le même preprocessing que dans les notebooks d'entraînement.


In [4]:
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw' / 'movielens' / 'ml-100k').resolve()
PROCESSED_PATH = (NOTEBOOK_ROOT / '..' / 'data' / 'processed' / 'movielens_100k' / 'interactions.parquet').resolve()

if PROCESSED_PATH.exists():
    df = pd.read_parquet(PROCESSED_PATH)
    print('Loaded processed dataset:', PROCESSED_PATH)
else:
    df = load_movielens_100k(RAW_ROOT)
    df = encode_ids(df)
    print('Loaded raw MovieLens 100K and encoded IDs')

if 'label' not in df.columns:
    df['label'] = (df['rating'] >= 4).astype(int)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)
print('Unique users:', df['user_id_encoded'].nunique())
print('Unique items:', df['item_id_encoded'].nunique())

Loaded processed dataset: /home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_100k/interactions.parquet
train shape: (79429, 20)
test shape: (19858, 20)
Unique users: 943
Unique items: 1349


## Chargement et preprocessing - MovieLens 1M

(Section ajoutée pour charger et préparer le jeu MovieLens 1M sans modifier les cellules existantes.)

In [7]:
# MovieLens 1M preprocessing
ml1m_path = (NOTEBOOK_ROOT / '..' / 'data' / 'processed' / 'movielens_1m' / 'interactions.parquet').resolve()
if ml1m_path.exists():
    df_ml_1m = pd.read_parquet(ml1m_path)
    print('Loaded processed ML-1M:', ml1m_path)
else:
    raw_ml1m = (NOTEBOOK_ROOT / '..' / 'data' / 'raw' / 'movielens' / 'ml-1m').resolve()
    try:
        from data_utils import load_movielens_1m
        df_ml_1m = load_movielens_1m(raw_ml1m)
        df_ml_1m = encode_ids(df_ml_1m)
        print('Loaded raw ML-1M and encoded IDs')
    except Exception as e:
        print('Could not load ML-1M raw data:', e)
        df_ml_1m = pd.DataFrame()

if not df_ml_1m.empty:
    df_ml_1m = extract_all_context_features(df_ml_1m)
    if 'label' not in df_ml_1m.columns:
        if 'rating' in df_ml_1m.columns:
            df_ml_1m['label'] = (df_ml_1m['rating'] >= 4).astype(int)
        else:
            df_ml_1m['label'] = 1
    from sklearn.model_selection import train_test_split
    if df_ml_1m['label'].nunique() > 1:
        strat = df_ml_1m['label']
    else:
        strat = None
    train_df_ml_1m, test_df_ml_1m = train_test_split(df_ml_1m, test_size=0.2, random_state=42, stratify=strat)
    print('ML-1M train/test shapes:', train_df_ml_1m.shape, test_df_ml_1m.shape)
else:
    print('ML-1M dataframe empty, skipping preprocessing')


Loaded processed ML-1M: /home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_1m/interactions.parquet
ML-1M train/test shapes: (799688, 20) (199923, 20)


## Chargement et preprocessing - RetailRocket

(Section ajoutée pour charger et préparer le jeu RetailRocket.)

In [8]:
# RetailRocket preprocessing
rr_path = (NOTEBOOK_ROOT / '..' / 'data' / 'processed' / 'retailrocket' / 'interactions.parquet').resolve()
if rr_path.exists():
    df_retail = pd.read_parquet(rr_path)
    print('Loaded processed RetailRocket:', rr_path)
else:
    print('RetailRocket processed file not found:', rr_path)
    df_retail = pd.DataFrame()

if not df_retail.empty:
    df_retail = extract_all_context_features(df_retail, is_retail_rocket=True)
    if 'label' not in df_retail.columns:
        if 'rating' in df_retail.columns:
            df_retail['label'] = (df_retail['rating'] >= 4).astype(int)
        else:
            df_retail['label'] = 1
    if df_retail['label'].nunique() > 1:
        strat = df_retail['label']
    else:
        strat = None
    train_df_retail, test_df_retail = train_test_split(df_retail, test_size=0.2, random_state=42, stratify=strat)
    print('RetailRocket train/test shapes:', train_df_retail.shape, test_df_retail.shape)
else:
    print('RetailRocket dataframe empty, skipping preprocessing')


Loaded processed RetailRocket: /home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/retailrocket/interactions.parquet
RetailRocket train/test shapes: (625215, 22) (156304, 22)


## Définitions de métriques et d'évaluation

Fonctions pour calculer les métriques de ranking sur un sous-ensemble d'exemples.


In [9]:
def evaluate_ranking(model, df, n_items, context_cols=None, k=10, sample_size=500, device=None):
    model.eval()
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    items = torch.arange(n_items, dtype=torch.long, device=device)
    eval_df = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True)
    hits, ndcgs, mrrs = [], [], []
    with torch.no_grad():
        for _, row in eval_df.iterrows():
            user_id = torch.tensor([row['user_id_encoded']], dtype=torch.long, device=device)
            item_id = int(row['item_id_encoded'])
            user_batch = user_id.expand(n_items)
            if context_cols is None:
                logits = model(user_batch, items).cpu().numpy()
            else:
                context = torch.tensor(row[context_cols].to_numpy(dtype=np.float32), dtype=torch.float32, device=device).unsqueeze(0)
                context_batch = context.expand(n_items, len(context_cols))
                logits = model(user_batch, items, context_batch).cpu().numpy()
            ranking = np.argsort(-logits)
            hits.append(int(item_id in ranking[:k]))
            ndcgs.append(ndcg(ranking, item_id, k))
            mrrs.append(mrr(ranking, item_id))
    return np.mean(hits), np.mean(ndcgs), np.mean(mrrs)


## Chargement des modèles

Nous reconstruisons les architectures et chargeons les poids sauvegardés.


In [10]:
n_users = df['user_id_encoded'].nunique()
n_items = df['item_id_encoded'].nunique()
context_cols = [
    'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos',
    'is_weekend',
    'time_since_last_interaction_log',
    'session_length',
    'session_position_norm'
]
context_dim = len(context_cols)

baseline_model = None
context_model = None

baseline_n_items = None
context_n_items = None

if baseline_exists:
    baseline_state = torch.load(BASELINE_PATH, map_location='cpu')
    baseline_n_items = baseline_state['embedding_item_gmf.weight'].shape[0]
    baseline_n_users = baseline_state['embedding_user_gmf.weight'].shape[0]
    print(f'Baseline checkpoint: {baseline_n_users} users, {baseline_n_items} items')

    baseline_model = NCF(
        num_users=baseline_n_users,
        num_items=baseline_n_items,
        embed_dim=32,
        mlp_layers=[64, 32, 16],
        dropout=0.2
    )
    baseline_model.load_state_dict(baseline_state)
    baseline_model.to('cpu')
    print('Baseline model loaded.')

if context_exists:
    context_state = torch.load(CONTEXT_PATH, map_location='cpu')
    context_n_items = context_state['ncf_core.embedding_item_gmf.weight'].shape[0]
    context_n_users = context_state['ncf_core.embedding_user_gmf.weight'].shape[0]
    print(f'Context checkpoint: {context_n_users} users, {context_n_items} items')

    context_model = NCFContext(
        num_users=context_n_users,
        num_items=context_n_items,
        context_dim=context_dim,
        embed_dim=32,
        mlp_layers=[64, 32, 16],
        context_embed_dim=32,
        dropout=0.2,
        fusion_type='concat'
    )
    context_model.load_state_dict(context_state)
    context_model.to('cpu')
    print('Context model loaded.')


Baseline checkpoint: 943 users, 1349 items
Baseline model loaded.
Context checkpoint: 943 users, 1682 items
Context model loaded.


## Chargement des checkpoints - ML-1M et RetailRocket

In [11]:
# Try to load ML-1M and RetailRocket checkpoints if present
BASELINE_PATH_ML1M = RESULTS_DIR / 'baseline_ncf_ml1m.pt'
CONTEXT_PATH_ML1M = RESULTS_DIR / 'context_ncf_ml1m.pt'
BASELINE_PATH_RR = RESULTS_DIR / 'baseline_ncf_retailrocket.pt'
CONTEXT_PATH_RR = RESULTS_DIR / 'context_ncf_retailrocket.pt'

baseline_exists_ml1m = BASELINE_PATH_ML1M.exists()
context_exists_ml1m = CONTEXT_PATH_ML1M.exists()
baseline_exists_rr = BASELINE_PATH_RR.exists()
context_exists_rr = CONTEXT_PATH_RR.exists()

print('ML1M baseline exists:', baseline_exists_ml1m, 'context exists:', context_exists_ml1m)
print('RetailRocket baseline exists:', baseline_exists_rr, 'context exists:', context_exists_rr)

baseline_model_ml1m = None
context_model_ml1m = None
baseline_n_items_ml1m = None
context_n_items_ml1m = None

baseline_model_rr = None
context_model_rr = None
baseline_n_items_rr = None
context_n_items_rr = None

if baseline_exists_ml1m:
    state = torch.load(BASELINE_PATH_ML1M, map_location='cpu')
    if 'embedding_item_gmf.weight' in state:
        baseline_n_items_ml1m = state['embedding_item_gmf.weight'].shape[0]
    if 'embedding_user_gmf.weight' in state:
        baseline_n_users_ml1m = state['embedding_user_gmf.weight'].shape[0]
    print('Baseline ML1M checkpoint items/users:', baseline_n_items_ml1m, baseline_n_users_ml1m)
    baseline_model_ml1m = NCF(
        num_users=baseline_n_users_ml1m,
        num_items=baseline_n_items_ml1m,
        embed_dim=32,
        mlp_layers=[64,32,16],
        dropout=0.2
    )
    baseline_model_ml1m.load_state_dict(state)
    baseline_model_ml1m.to('cpu')

if context_exists_ml1m:
    cstate = torch.load(CONTEXT_PATH_ML1M, map_location='cpu')
    # context state structure may differ; try to find embedding shapes
    if 'ncf_core.embedding_item_gmf.weight' in cstate:
        context_n_items_ml1m = cstate['ncf_core.embedding_item_gmf.weight'].shape[0]
    if 'ncf_core.embedding_user_gmf.weight' in cstate:
        context_n_users_ml1m = cstate['ncf_core.embedding_user_gmf.weight'].shape[0]
    print('Context ML1M checkpoint items/users:', context_n_items_ml1m, context_n_users_ml1m)
    context_model_ml1m = NCFContext(
        num_users=context_n_users_ml1m,
        num_items=context_n_items_ml1m,
        context_dim=context_dim,
        embed_dim=32,
        mlp_layers=[64,32,16],
        context_embed_dim=32,
        dropout=0.2,
        fusion_type='concat'
    )
    context_model_ml1m.load_state_dict(cstate)
    context_model_ml1m.to('cpu')

if baseline_exists_rr:
    state_rr = torch.load(BASELINE_PATH_RR, map_location='cpu')
    if 'embedding_item_gmf.weight' in state_rr:
        baseline_n_items_rr = state_rr['embedding_item_gmf.weight'].shape[0]
    if 'embedding_user_gmf.weight' in state_rr:
        baseline_n_users_rr = state_rr['embedding_user_gmf.weight'].shape[0]
    baseline_model_rr = NCF(
        num_users=baseline_n_users_rr,
        num_items=baseline_n_items_rr,
        embed_dim=32,
        mlp_layers=[64,32,16],
        dropout=0.2
    )
    baseline_model_rr.load_state_dict(state_rr)
    baseline_model_rr.to('cpu')

if context_exists_rr:
    cstate_rr = torch.load(CONTEXT_PATH_RR, map_location='cpu')
    if 'ncf_core.embedding_item_gmf.weight' in cstate_rr:
        context_n_items_rr = cstate_rr['ncf_core.embedding_item_gmf.weight'].shape[0]
    if 'ncf_core.embedding_user_gmf.weight' in cstate_rr:
        context_n_users_rr = cstate_rr['ncf_core.embedding_user_gmf.weight'].shape[0]
    context_model_rr = NCFContext(
        num_users=context_n_users_rr,
        num_items=context_n_items_rr,
        context_dim=context_dim,
        embed_dim=32,
        mlp_layers=[64,32,16],
        context_embed_dim=32,
        dropout=0.2,
        fusion_type='concat'
    )
    context_model_rr.load_state_dict(cstate_rr)
    context_model_rr.to('cpu')


ML1M baseline exists: True context exists: True
RetailRocket baseline exists: True context exists: True
Baseline ML1M checkpoint items/users: 3416 6040
Context ML1M checkpoint items/users: 3416 6040


## Évaluation et tableau de comparaison

Nous calculons les métriques pour chaque modèle et affichons les gains.


In [12]:
results = []
if baseline_model is not None:
    hit10, ndcg10, mrr_score = evaluate_ranking(
        baseline_model,
        test_df,
        baseline_n_items,
        context_cols=None,
        k=10,
        sample_size=500,
        device='cpu'
    )
    results.append({
        'model': 'baseline',
        'hit10': hit10,
        'ndcg10': ndcg10,
        'mrr': mrr_score
    })

if context_model is not None:
    hit10, ndcg10, mrr_score = evaluate_ranking(
        context_model,
        test_df,
        context_n_items,
        context_cols=context_cols,
        k=10,
        sample_size=500,
        device='cpu'
    )
    results.append({
        'model': 'context',
        'hit10': hit10,
        'ndcg10': ndcg10,
        'mrr': mrr_score
    })

results_df = pd.DataFrame(results)
results_df = results_df[['model', 'hit10', 'ndcg10', 'mrr']]
display(results_df)


,model,hit10,ndcg10,mrr
0,baseline,0.018,0.011961,0.015754
1,context,0.018,0.008479,0.010267


## Évaluation - ML-1M et RetailRocket

(Cells ajoutées pour évaluer les modèles chargés sur ML-1M et RetailRocket et compléter le tableau de résultats.)

In [17]:
# Evaluate ML-1M if models and data available
if 'test_df_ml_1m' in globals():
    if baseline_model_ml1m is not None:
        hit10, ndcg10, mrr_score = evaluate_ranking(baseline_model_ml1m, test_df_ml_1m, baseline_n_items_ml1m, context_cols=None, k=10, sample_size=500, device='cpu')
        results.append({'model': 'baseline', 'dataset': 'ml-1m', 'hit10': hit10, 'ndcg10': ndcg10, 'mrr': mrr_score})
    if context_model_ml1m is not None:
        hit10, ndcg10, mrr_score = evaluate_ranking(context_model_ml1m, test_df_ml_1m, context_n_items_ml1m, context_cols=context_cols, k=10, sample_size=500, device='cpu')
        results.append({'model': 'context', 'dataset': 'ml-1m', 'hit10': hit10, 'ndcg10': ndcg10, 'mrr': mrr_score})
else:
    print('test_df_ml_1m not found; skipping ML-1M evaluation')

# Evaluate RetailRocket if models and data available
if 'test_df_retail' in globals() or 'test_df_retailrocket' in globals():
    test_rr = globals().get('test_df_retail', globals().get('test_df_retailrocket'))
    if baseline_model_rr is not None:
        hit10, ndcg10, mrr_score = evaluate_ranking(baseline_model_rr, test_rr, baseline_n_items_rr, context_cols=None, k=10, sample_size=500, device='cpu')
        results.append({'model': 'baseline', 'dataset': 'retailrocket', 'hit10': hit10, 'ndcg10': ndcg10, 'mrr': mrr_score})
    if context_model_rr is not None:
        hit10, ndcg10, mrr_score = evaluate_ranking(context_model_rr, test_rr, context_n_items_rr, context_cols=context_cols, k=10, sample_size=500, device='cpu')
        results.append({'model': 'context', 'dataset': 'retailrocket', 'hit10': hit10, 'ndcg10': ndcg10, 'mrr': mrr_score})
else:
    print('RetailRocket test dataframe not found; skipping RetailRocket evaluation')

# Merge with existing results_df if present
try:
    if 'results_df' in globals():
        extra_df = pd.DataFrame(results)
        if not extra_df.empty:
            results_df_all = pd.concat([results_df, extra_df], ignore_index=True, sort=False)
        else:
            results_df_all = results_df
    else:
        results_df_all = pd.DataFrame(results)
    display(results_df_all.tail(4))
except Exception as e:
    print('Error while aggregating results:', e)


,model,hit10,ndcg10,mrr,dataset
20,baseline,0.008,0.003921,0.006613,ml-1m
21,context,0.006,0.004036,0.006365,ml-1m
22,baseline,0.004,0.002774,0.003551,retailrocket
23,context,0.004,0.002602,0.003349,retailrocket


## Interprétation rapide

Si le modèle contextuel est meilleur, cela valide l'intérêt des features temporelles et de session. Si le score reste similaire, il faudra explorer l'architecture, la balance des données ou la qualité des features (prétraitement, bruit, granularité).